# LangGraph: 상태를 기억하며 Agent 흐름 제어하기

LLM(Large Language Model, 대규모 언어 모델) 호출은 입력을 받아 응답을 한 번 생성한다. 반면 **AI Agent**는 목표를 해결할 때까지 판단, Tool 실행, 결과 확인을 여러 번 이어 갈 수 있다. Tool(도구)은 검색, 계산, 데이터 조회처럼 모델 밖의 실제 작업을 수행하는 함수나 외부 시스템이다. 이 과정에서는 현재까지 모은 정보와 다음 실행 위치를 함께 관리해야 한다.

**LangGraph**는 이런 장기 실행·상태 기반 Agent를 그래프로 설계하고 실행하는 저수준 오케스트레이션 프레임워크이자 런타임이다. 모델 자체를 학습시키는 도구가 아니라 작업의 순서, 분기, 반복, 상태 저장과 사람의 승인 단계를 제어하는 도구이다.

## 왜 필요한가

Chain(체인)은 여러 처리 단계를 개발자가 정한 순서대로 연결한 실행 흐름이다. 정해진 순서만 실행한다면 일반 함수나 Chain으로도 충분하다. 하지만 모델 판단에 따라 Tool을 호출하고, 실패한 단계를 다시 시도하고, 중간에 사람의 승인을 기다려야 한다면 실행 흐름이 복잡해진다. LangGraph는 이 흐름을 **State, Node, Edge**로 드러내어 실행 경로를 추적하고 다시 시작하기 쉽게 만든다.

이번 장은 코드를 실행하지 않는 개념 노트북이다. 다음 장부터 같은 용어를 실제 Python 코드로 구현한다.


## LangChain과 LangGraph는 어떻게 연결되는가

LangChain과 LangGraph는 경쟁 도구가 아니라 서로 다른 층을 담당한다. LangChain은 Model, Prompt, Tool, Retriever와 표준 Agent 같은 상위 수준 구성 요소를 제공하고, LangGraph는 그 구성 요소를 어떤 상태와 경로로 실행할지 제어한다. 현재 LangChain의 Agent도 내부 실행 기반으로 LangGraph를 사용하지만, LangGraph를 LangChain 없이 사용할 수도 있다.

![LangChain과 LangGraph의 관계](https://cdn.jsdelivr.net/gh/goat-skn-ai/image-repo@7f008bd902003af9b0138e956af0be37ffa7a9ff/08_llm/08_langgraph/01_langgraph_basics/langchain_langgraph_relationship.png)

그림은 왼쪽에서 오른쪽으로 읽는다.

1. 왼쪽의 LangChain은 Model, Prompt, Tool처럼 Node 안에서 사용할 작업 부품을 제공한다.
2. 오른쪽의 LangGraph는 State를 공유하면서 Node의 순서, 분기, 반복과 종료를 제어한다.
3. 오른쪽 주변의 State, Persistence, Human-in-the-loop, Streaming은 LangGraph가 상태 저장·중단·재개·실시간 출력 같은 실행 기능을 담당한다는 뜻이다.
4. 아래의 양방향 점선은 두 도구 중 하나만 고르는 관계가 아니라, 요구 수준에 따라 LangChain만 사용하거나 LangGraph를 함께 사용할 수 있다는 뜻이다.
5. 빠르게 일반 Agent를 만들 때는 LangChain의 상위 API가 편하고, 실행 경로를 세밀하게 설계할 때는 LangGraph를 함께 사용한다.

> 이 관계도는 [LangGraph 공식 개요](https://docs.langchain.com/oss/python/langgraph/overview)의 제품 역할 설명을 바탕으로 제작한 수업용 이미지이다.


## Agent는 왜 반복 구조가 필요한가

고정 Workflow는 개발자가 실행 순서를 미리 정한다. Agent는 현재 상황을 본 LLM이 다음 행동과 Tool 사용 여부를 결정하므로 실행 경로가 입력과 Tool 결과에 따라 달라진다.

![LangGraph 공식 Agent Workflow](https://cdn.jsdelivr.net/gh/goat-skn-ai/image-repo@7f008bd902003af9b0138e956af0be37ffa7a9ff/08_llm/08_langgraph/01_langgraph_basics/langgraph_official_agent_loop.png)

공식 그림은 다음 순서로 읽는다.

1. `In`: 사용자의 질문이 들어온다.
2. `LLM call`: 모델이 바로 답할지 Tool을 사용할지 판단한다.
3. `action → Tool`: Tool이 검색이나 계산 같은 실제 작업을 수행한다.
4. `feedback`: Tool 결과가 모델로 돌아가 다음 판단의 입력이 된다.
5. `Out`: 충분한 근거가 모이면 반복을 끝내고 답변을 반환한다.

LangGraph에서는 LLM과 Tool을 Node로, 두 단계 사이의 이동을 Edge로, 질문과 Tool 결과를 State로 표현한다. 따라서 그림의 feedback 화살표가 그래프의 **반복 Edge**에 해당한다.

> 이미지 출처: [LangChain 공식 문서 - Workflows and agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents)


## State, Node, Edge로 실행 흐름 표현하기

예를 들어 사용자가 날씨를 물으면 그래프는 질문을 기억하고, 모델이 Tool 사용 여부를 판단하며, 날씨 Tool의 결과를 다시 모델에 전달해야 한다. 이때 세 구성 요소는 서로 다른 질문에 답한다.

### State — 무엇을 기억하는가

State(상태)는 그래프가 실행되는 동안 Node가 함께 읽고 갱신하는 데이터이다. 사용자 메시지, Tool 결과, 검색 문서와 처리 완료 여부 등을 담는다.

### Node — 무슨 일을 하는가

Node(노드)는 State를 입력받아 작업한 뒤 State의 변경분을 반환하는 실행 단위이다. LLM 호출, Tool 실행, 데이터 검증과 사람의 승인 단계가 모두 Node가 될 수 있으며, Node가 반드시 Agent일 필요는 없다.

### Edge — 다음에 어디로 이동하는가

Edge(간선)는 한 Node가 끝난 뒤 실행할 다음 Node를 정한다. 목적지가 항상 같으면 고정 Edge를, State 값에 따라 경로가 달라지면 Conditional Edge(조건부 간선)를 사용한다.

`START → 모델 Node → Tool 필요 여부 판단 → Tool Node 또는 END`

`START`는 입력 State를 첫 Node로 보내는 가상 시작점이고, `END`는 더 실행할 Node가 없다는 가상 종료점이다.


## 그래프 설계와 실제 실행은 다른 단계이다

`StateGraph`는 State 구조를 기준으로 Node와 Edge를 조립하는 Builder(설계 객체)이다. 먼저 설계도를 완성한 뒤 `compile()`로 실행 가능한 그래프를 만들고, 마지막에 `invoke()`로 실제 입력을 처리한다. 아래 코드는 API의 연결 순서만 보여 주는 축약 예시이며 이 셀에서는 실행하지 않는다.

```python
builder = StateGraph(State)          # 1. State를 사용하는 설계 객체
builder.add_node('chatbot', chatbot) # 2. 실행할 Node 등록
builder.add_edge(START, 'chatbot')   # 3. 이동 경로 연결
builder.add_edge('chatbot', END)

graph = builder.compile()            # 4. 실행 가능한 그래프로 변환
result = graph.invoke(initial_state) # 5. 실제 입력을 넣고 최종 State 반환
```

### `compile()` — 설계도를 실행 가능한 그래프로 바꾼다

Node와 Edge 연결을 검사하고 실행에 필요한 구조를 만든다. 이 시점에는 아직 사용자의 질문을 처리하지 않는다.

### `invoke(initial_state)` — 그래프를 한 번 실행한다

초기 State를 받아 `START`부터 `END`까지 이동한 뒤 갱신된 최종 State를 반환한다. 다음 노트북에서 이 축약 코드를 실행 가능한 채팅 그래프로 완성한다.


## Reducer는 State 갱신 규칙이다

여러 Node가 같은 State 필드를 갱신하면 기존 값과 새 값을 어떻게 합칠지 정해야 한다. **Reducer(리듀서)** 는 Node가 반환한 변경값을 기존 State에 적용하는 규칙이다.

| 단계 | `messages` 값 |
| --- | --- |
| 기존 State | `[사용자 질문]` |
| Node가 반환한 변경값 | `[AI 답변]` |
| Reducer 없음 | `[AI 답변]`으로 덮어쓴다. |
| `add_messages` 사용 | `[사용자 질문, AI 답변]`으로 병합한다. |

`MessagesState`는 `messages` 필드와 메시지 병합 규칙이 미리 준비된 State이다. 다음 채팅 실습은 `TypedDict`로 State를 직접 정의하고 `add_messages`를 연결해 이 병합 과정을 눈으로 확인한다.

> `MessageGraph`는 State 전체를 메시지 목록 하나로 제한한 deprecated(사용 중단 예정) API이다. 새 코드에서는 `StateGraph`와 `MessagesState` 또는 사용자 정의 State를 사용한다.


## 조건부 분기로 Tool 호출 반복 만들기

Tool을 사용하는 Agent는 모델의 판단에 따라 다음 경로가 달라진다. 모델이 Tool을 요청하면 Tool Node로 이동하고, 요청하지 않으면 `END`로 이동한다. Tool 실행 뒤에는 다시 모델 Node로 돌아가 결과를 확인하므로 그래프에 순환이 생긴다.

`사용자 질문 → 모델 판단 → Tool 요청 → Tool 실행 → 모델 재판단 → 최종 답변`

여기서 이름이 비슷한 두 값을 구분해야 한다.

- **`AIMessage.tool_calls`**: 모델이 만든 함수 이름·인자·호출 ID가 담긴 **실행 요청**이다.
- **`ToolMessage`**: Tool Node가 함수를 실제 실행한 뒤 만든 **실행 결과**이다. 요청의 호출 ID와 연결된다.

다음 `03_langgraph_chatbot_with_tools.ipynb`에서 Conditional Edge가 두 경로를 선택하고, Tool 결과가 다시 모델로 전달되는 과정을 구현한다.


## 이후 단원에서 확장할 기능

State, Node, Edge가 기본 골격이고 나머지 기능은 이 골격 위에 추가된다. 지금은 이름을 외우기보다 어떤 문제를 해결하는지만 구분한다.

### 기억하고 다시 시작하기

Checkpoint는 단계별 State와 실행 위치를 저장하고, `thread_id`는 이어서 실행할 대화를 구분한다. 실패 후 복구하거나 이전 대화를 계속할 때 사용한다.

### 사람의 승인 기다리기

`interrupt()`는 그래프를 잠시 멈추고, `Command(resume=값)`은 사람의 입력을 전달해 중지한 위치부터 다시 실행한다. 중요한 작업 전에 승인받는 Human-in-the-loop에 사용한다.

### 여러 Agent로 역할 나누기

Multi-Agent는 역할이 다른 Agent가 State를 공유하거나 결과를 넘겨 하나의 목표를 해결하는 구조이다. 역할 분리가 필요할 때만 사용하며, Agent 수가 늘면 모델 호출 비용과 종료 조건 관리도 복잡해진다.

이 단원의 핵심은 한 문장으로 정리할 수 있다. **LangGraph는 State를 공유하는 Node들을 Edge로 연결하여 분기·반복·중단·재개가 가능한 Agent 실행 흐름을 만든다.**
